# T10 It6: the pre-registered finalists on the fresh NUDT 808 draw, scored once

Reads `results/preregistered_t10.json` (written before this set was opened) and applies it unchanged. Primary: NUDT 808 at x1. Robustness:
every inlier-count signal scaled x0.5 ... x2. Reported: score, wrong, answered, queries, paired bootstrap vs the base, the plain bar that
answers as many photos, `answer_change`; a configuration manifest per rule.

In [ ]:
import os, sys, json, hashlib, time, gc
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.decision_signals import build_signal_table
from train.checks.holdout import NUDT_DESIGN, answer_change, bar_frontier, load_t8_table, versus_frontier
from train.checks.eval_tools import failure_log, manifest, save_manifest
RES = ROOT / "tests/Test_Iteration_10/results"
PRE_PATH = RES / "preregistered_t10.json"; PRE = json.loads(PRE_PATH.read_text())
print("pre-registration written", PRE["written"], "sha256", hashlib.sha256(PRE_PATH.read_bytes()).hexdigest())
(RES / "signals").mkdir(exist_ok=True)
out = RES / "signals" / "nudt808.csv.gz"
if not out.exists():
    build_signal_table(RES / "rich_dump" / "nudt_fresh808", "nudt808", int(os.environ.get("WORKERS", "4"))).to_csv(out, index=False)
N808 = cj.add_context(cj.add_features(pd.read_csv(out)))
D = load_t8_table(ROOT, locked=True); D = D[D.data.isin(NUDT_DESIGN)].reset_index(drop=True)
print(N808.photo.nunique(), "photos in NUDT 808")

In [ ]:
F = cj.FEATURES_CTX; FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]
SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
def models(var):
    m = None if var == "pc" else 5
    d = D if m is None else cj.add_context(D, m=m)
    _, judge = cj.judge_probability(d, NUDT_DESIGN, F, "gbm")
    v5 = PRE["finalists"][var]["V5"]; prior = v5["prior"]
    v5model, _, st = sr.fit_judge(d, NUDT_DESIGN, FR, {"rli", "z", "hull", "cover9"}, v5["augment"], lambda x: sr.add_relative(x, prior, 30, m))
    del st; gc.collect()
    return m, judge, v5model, prior
def rules(var, t, m, judge, v5model, prior):
    fin = PRE["finalists"][var]
    jk = dict(jetson=True, m=5) if var == "jetson" else {}
    p = sr.predict(judge, t, F)
    out = {"base": cj.baseline(t, 8, 11, min_choice=3) if var == "pc" else cj.baseline(t, 8, 8, jetson=True, m=5)}
    for k in ["J0", "J1", "J2"]:
        f = fin[k]
        out[k] = cj.decide(t, p, f["cut"] if f["cut"] is not None else cj.NO_ANSWER, override=f["override"], wide_tau=f["wide_cut"],
                           min_choice=f["min_choice"], early_tau=f["early_exit"], **jk)
    tr = sr.add_relative(t, prior, 30, m); f = fin["V5"]
    out["V5"] = cj.decide(tr, sr.predict(v5model, tr, FR), f["cut"] if f["cut"] is not None else cj.NO_ANSWER, override=None, wide_tau=f["wide_cut"],
                          min_choice=f["min_choice"], early_tau=f["early_exit"], **jk)
    return out
t0 = time.time(); RES808 = {}; ROB = []
for var in ["pc", "jetson"]:
    m, judge, v5model, prior = models(var)
    for s in SCALES:
        t = sr.scale_signals(N808, s, m=m)
        r = rules(var, t, m, judge, v5model, prior)
        if s == 1.0: RES808[var] = (t, r)
        for k, v in r.items():
            sm = cj.summary(v).loc["nudt808"]
            ROB.append(dict(var=var, rule=k, scale=s, score=sm.score, wrong=int(sm.wrong), answered=sm.answered, queries=sm.queries))
ROB = pd.DataFrame(ROB); ROB.to_csv(RES / "it6_808_scales.csv", index=False)
print("%.0f s" % (time.time() - t0))

## Primary: NUDT 808 at x1

In [ ]:
FRONT = bar_frontier(RES808["pc"][0]); FRONT.to_csv(RES / "it6_808_bar_frontier.csv", index=False)
rows = []
for var, (t, r) in RES808.items():
    base = r["base"]
    for k, v in r.items():
        g = cj.paired_gain(base, v, ["nudt808"])["nudt808"]
        vf = versus_frontier(v, FRONT, var).iloc[0]; a = answer_change(base, v).iloc[0]
        rows.append(dict(var=var, rule=k, score=vf.score, wrong=vf.wrong, answered=vf.n_answered, queries=vf.queries, gain=g, above_bar=vf.above_bar,
                         same_n_bar=vf.same_n_bar, bar_wrong=vf.bar_wrong, added_right=a.added_right, added_wrong=a.added_wrong, removed=a.removed,
                         removed_wrong=a.removed_wrong))
        fin = PRE["finalists"][var].get(k)
        save_manifest(RES / f"it6_808_{var}_{k}.csv", manifest(var, cut=None if k == "base" else fin["cut"], wide_cut=None if k == "base" else fin["wide_cut"],
                      override=None if k == "base" else fin["override"], early_exit=None if k == "base" else fin["early_exit"],
                      bars=[8, 11] if (k == "base" and var == "pc") else ([8, 8] if k == "base" else None), min_choice=1 if k == "base" else 3,
                      judge_features=None if k == "base" else ("relative + augmentation" if k == "V5" else "FEATURES_CTX"),
                      judge_trained_on=None if k == "base" else NUDT_DESIGN, data="nudt808", note="pre-registered T10"))
        v.to_csv(RES / f"it6_808_{var}_{k}.csv", index=False)
FINAL = pd.DataFrame(rows); FINAL.to_csv(RES / "it6_808_final.csv", index=False); FINAL

## Robustness: wrong answers and score on NUDT 808 with every inlier signal scaled

In [ ]:
pd.concat({"wrong": ROB.pivot_table(index=["var", "rule"], columns="scale", values="wrong").astype(int),
           "score": ROB.pivot_table(index=["var", "rule"], columns="scale", values="score").round(2)}, axis=1)

## Failure log on NUDT 808 (x1)

In [ ]:
parts = [failure_log(t, v, var).assign(var=var, rule=k) for var, (t, r) in RES808.items() for k, v in r.items()]
FL = pd.concat(parts); FL.to_csv(RES / "it6_808_failure_log.csv", index=False)
FL.groupby(["var", "rule"]).outcome.value_counts().unstack(fill_value=0)

## Against the pre-registered criteria

In [ ]:
V = []
for (var, k), g in FINAL.set_index(["var", "rule"]).iterrows():
    if k == "base": continue
    base_w = int(FINAL[(FINAL["var"] == var) & (FINAL.rule == "base")].wrong.iloc[0])
    lo = float(g.gain.split("[")[1].split(",")[0]); mean = float(g.gain.split(" ")[0])
    w_scales = ROB[(ROB["var"] == var) & (ROB.rule == k)].wrong
    j0_scales = ROB[(ROB["var"] == var) & (ROB.rule == "J0")].wrong.sum()
    V.append(dict(var=var, rule=k, wrong_x1=int(g.wrong), base_wrong=base_w, gain=g.gain,
                  counts_as_gain=bool(g.wrong == 0 and g.wrong <= base_w and mean >= 0 and lo > 0),
                  wrong_over_scales=int(w_scales.sum()), scale_robust=bool(w_scales.sum() == 0), safer_than_J0=bool(w_scales.sum() < j0_scales)))
VERDICT = pd.DataFrame(V); VERDICT.to_csv(RES / "it6_808_verdict.csv", index=False); VERDICT